In [ ]:
from pathlib import Path

import polars as pl
from loguru import logger

from scrapetube.utils.config import (
    get_timestamp_string
)
from scrapetube.cc.text import convert_subtitles_to_text, export_dataframe_to_jsonl
from scrapetube.cc.license import check_video_licenses_concurrent
from scrapetube.utils.logger import setup_logging

pl.Config.set_fmt_str_lengths(100)

today_string = get_timestamp_string()
setup_logging(write_to_file=False)
data_dir = Path("../data")

In [ ]:
def load_latest_data(prefix:str, data_dir:Path):
    """Load the most recently modified parquet file with the given prefix from data directory."""
    latest_file = max(data_dir.glob(f"{prefix}*.parquet"), key=lambda f: f.stat().st_mtime)
    logger.info(f"Loaded {latest_file}")
    df = pl.read_parquet(latest_file)
    return df

In [ ]:
subtitle_df = load_latest_data("subtitle", data_dir)
subtitle_df

In [ ]:
channel_video_meta_df = load_latest_data("channel_video_meta", data_dir)

In [ ]:
channel_df_with_sub = channel_video_meta_df.join(
    subtitle_df, on="video_id", how="inner"
).with_columns(pl.col("subtitles").str.json_decode().alias("subtitles_json"))
channel_df_with_sub = convert_subtitles_to_text(channel_df_with_sub)

In [ ]:
check_license_result = check_video_licenses_concurrent(
    video_urls=channel_df_with_sub["video_url"].to_list(),
    sleep=(5, 25),
)

In [ ]:
channel_df_with_sub = channel_df_with_sub.with_columns(
    pl.Series([item["license"] for item in check_license_result]).alias("license")
).filter(
    pl.col("license")
    .str.strip_chars()
    .str.to_lowercase()
    .str.contains("creative commons")
)

In [ ]:
export_dataframe_to_jsonl(channel_df_with_sub)